# Sequence Place Recognition Benchmark Report

This report benchmarks the impact of sequence length (window size) across multiple maps. It shows per-map metrics over window sizes and overlays the cross-map mean. Aggregated mean and weighted-mean plots are also provided.


## Configuration

Paths, constants, and controls used throughout the report.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Data paths
DB_INDEX_DIR = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed/map1/keyframe_map"
)
ROOT_DATA_DIR = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed"
)

assert DB_INDEX_DIR.exists(), f"Path {DB_INDEX_DIR} does not exist"
assert ROOT_DATA_DIR.exists(), f"Path {ROOT_DATA_DIR} does not exist"

# Experiments root
EXP_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition/experiments/imageseq_benchmarks")
EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Maps to evaluate
MAPS = [f"map{i}" for i in range(2, 9)]

# Device and hyperparameters
DEVICE = "cuda"
PER_FRAME_K = 100
FINAL_K = 25
PR_PC_QUANTIZATION_SIZE = 0.05
RECALL_THRESHOLD_M = 3.0

# Controls
FORCE_RERUN = True
SKIP_IF_EXISTS = False

# Sweep settings
SEQ_LENGTHS = list(range(1, 3))


## Helpers

Utility functions to build PR caches and run the sequence benchmark sweep.


In [8]:
from __future__ import annotations
from typing import Iterable
import json
from pathlib import Path
from IPython.display import display
from tqdm import tqdm
from typing import Sequence

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from mmpr.pr_infer import PRInferConfig, PRInferencer
from mmpr.seq_pr_benchmark import SequenceBenchmarkConfig, SequencePRBenchmarker


def build_pr_cache_for_map(map_name: str) -> Path:
    """Build or load PR cache for a given map.

    Args:
        map_name: Map identifier such as "map2".
    Returns:
        Path to NPZ cache file.
    """
    pr_cache_path = Path(
        f"/home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_{map_name}.npz"
    )
    if pr_cache_path.exists() and not FORCE_RERUN:
        print(f"Using existing PR cache: {pr_cache_path}")
        return pr_cache_path

    cfg_inf = PRInferConfig(
        root_data_dir=ROOT_DATA_DIR,
        map_name=map_name,
        db_map_dir=DB_INDEX_DIR,
        index_dir=DB_INDEX_DIR,
        device=DEVICE,
        per_frame_k=PER_FRAME_K,
        pr_quant_size=PR_PC_QUANTIZATION_SIZE,
    )
    PRInferencer(cfg_inf, model="megaloc").save(pr_cache_path)
    print(f"Built PR cache: {pr_cache_path}")
    return pr_cache_path


def _configs_match_json(d: dict, cfg: SequenceBenchmarkConfig) -> bool:
    """Return True if saved metrics.json config matches the benchmark config."""
    conf = d.get("config", {})
    try:
        return (
            int(conf.get("max_window", -1)) == int(cfg.max_window)
            and int(conf.get("per_frame_k_used", -1)) == int(cfg.per_frame_k_used)
            and int(conf.get("final_k", -1)) == int(cfg.final_k)
            and str(conf.get("recency_weighting", "")) == str(cfg.recency_weighting)
            and abs(float(conf.get("recall_threshold_m", -1.0)) - float(cfg.recall_threshold_m)) < 1e-9
        )
    except Exception:
        return False


def _row_from_metrics_json(d: dict, W: int, map_name: str) -> dict:
    """Convert metrics.json payload to a single summary row."""
    rk = d.get("recall_at_k", {}) or {}
    return {
        "w": int(W),
        "auc_pr": float(d.get("auc_pr", 0.0)),
        "f1_max": float(d.get("f1_max", 0.0)),
        "recall_at_1": float(rk.get("1", 0.0)),
        "recall_at_5": float(rk.get("5", 0.0)),
        "recall_at_10": float(rk.get("10", 0.0)),
        "recall_at_25": float(rk.get("25", 0.0)),
        "num_valid": int(d.get("num_queries_valid", 0)),
        "num_total": int(d.get("num_queries_total", 0)),
        "query_track": map_name,
    }


def run_sweep(
    map_name: str,
    pr_cache_path: Path,
    seq_lengths: Iterable[int] = SEQ_LENGTHS,
) -> pd.DataFrame:
    """Run or reuse sequence benchmark for a map across sequence lengths."""
    all_rows: list[dict] = []
    for W in tqdm(list(seq_lengths)):
        out_dir = EXP_ROOT / f"{map_name}_w{W:03d}"
        out_dir.mkdir(parents=True, exist_ok=True)
        cfg_b = SequenceBenchmarkConfig(
            db_index_dir=DB_INDEX_DIR,
            cache_path=pr_cache_path,
            root_data_dir=ROOT_DATA_DIR,
            map_name=map_name,
            max_window=int(W),
            per_frame_k_used=PER_FRAME_K,
            final_k=FINAL_K,
            recency_weighting="none",
            recall_threshold_m=RECALL_THRESHOLD_M,
        )
        metrics_path = out_dir / "metrics.json"

        if SKIP_IF_EXISTS and metrics_path.exists() and not FORCE_RERUN:
            try:
                d = json.loads(metrics_path.read_text())
                if _configs_match_json(d, cfg_b):
                    all_rows.append(_row_from_metrics_json(d, W, map_name))
                    continue
            except Exception:
                pass

        bench = SequencePRBenchmarker(cfg_b)
        artifacts = bench.run()
        bench.save(artifacts, out_dir)
        all_rows.append({
            "w": int(W),
            "auc_pr": float(artifacts.auc_pr),
            "f1_max": float(artifacts.f1_max),
            "recall_at_1": float(artifacts.recall_at_k.get(1, 0.0)),
            "recall_at_5": float(artifacts.recall_at_k.get(5, 0.0)),
            "recall_at_10": float(artifacts.recall_at_k.get(10, 0.0)),
            "recall_at_25": float(artifacts.recall_at_k.get(25, 0.0)),
            "num_valid": int(artifacts.num_queries_valid),
            "num_total": int(artifacts.num_queries_total),
            "query_track": map_name,
        })

    df = pd.DataFrame(all_rows).sort_values("w").reset_index(drop=True)
    return df


## Run Benchmarks

Build or reuse caches, run sweeps for each map, and store per-map and combined summaries.


In [3]:
# Run benchmarks for configured maps; save per-map and combined summaries
summaries: dict[str, pd.DataFrame] = {}

for m in MAPS:
    print(f"=== {m} ===")
    cache_path = build_pr_cache_for_map(m)
    df_m = run_sweep(m, cache_path, seq_lengths=SEQ_LENGTHS)
    summaries[m] = df_m
    # Save per-map summary
    out_map_dir = EXP_ROOT / m
    out_map_dir.mkdir(parents=True, exist_ok=True)
    (out_map_dir / "summary.csv").write_text(df_m.to_csv(index=False))

# Combined summary across maps
summary_all = pd.concat(list(summaries.values()), ignore_index=True)
(EXP_ROOT / "summary_all.csv").write_text(summary_all.to_csv(index=False))

display(summary_all.head(3))
display(summary_all.tail(3))


=== map2 ===


Using cache found in /home/docker_mmpr/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main
/home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
INFO:dinov2:using MLP layer as FFN


Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map2.npz


100%|██████████| 2/2 [00:00<00:00,  6.22it/s]


=== map3 ===


...

Using cache found in /home/docker_mmpr/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main
INFO:dinov2:using MLP layer as FFN


Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map8.npz


100%|██████████| 2/2 [00:00<00:00,  2.99it/s]


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.875916,0.775680,0.786667,0.822857,0.843810,0.868571,525,605,map2
1,2,0.879031,0.778448,0.790476,0.826667,0.838095,0.866667,525,605,map2
2,1,0.816202,0.717422,0.934186,0.961609,0.974406,0.985375,547,628,map3


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
11,2,0.789646,0.687783,0.816191,0.901792,0.932979,0.952887,1507,1595,map7
12,1,0.778719,0.666404,0.796392,0.858247,0.879725,0.939863,1164,1188,map8
13,2,0.780584,0.668824,0.804124,0.859966,0.880584,0.939863,1164,1188,map8


## Visualization helpers

In [9]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics: Sequence[str] = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w' and metrics).
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = df["query_track"].iloc[0]
    # precompute simple mean once
    group = summary_all.groupby("w", as_index=False)
    mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()

    for m in metrics:
        if m not in df.columns:
            continue
        fig = px.line(df, x="w", y=m, title=f"{map_name}: {m} vs sequence length (w)", markers=True)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay simple mean across maps
        if m in mean_by_w.columns:
            fig.add_trace(
                go.Scatter(
                    x=mean_by_w["w"],
                    y=mean_by_w[m].astype(float),
                    mode="lines",
                    name="mean",
                    line=dict(color="green", dash="dash"),
                    showlegend=True,
                )
            )

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )
            fig.add_trace(
                go.Scatter(
                    x=wmean_series["w"],
                    y=wmean_series[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


In [10]:
def plot_aggregate_mean_median(
    summary_all,
    metrics: Sequence[str] = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Draw separate figures that show only mean and weighted mean across maps, with maxima highlighted.

    Args:
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    group = summary_all.groupby("w", as_index=False)
    mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()
    # Weighted mean by num_valid
    def _weighted_series(g):
        return pd.Series({k: float(np.average(g[k].astype(float), weights=g["num_valid"].astype(float))) for k in metrics if k in g.columns})
    wmean_by_w = summary_all.groupby("w").apply(_weighted_series, include_groups=False).reset_index()

    for m in metrics:
        if m not in summary_all.columns:
            continue
        fig = go.Figure()
        # Mean line
        fig.add_trace(
            go.Scatter(
                x=mean_by_w["w"],
                y=mean_by_w[m].astype(float),
                mode="lines",
                name="mean",
                line=dict(color="green", dash="dash"),
                showlegend=True,
            )
        )
        # Weighted mean line
        if m in wmean_by_w.columns:
            fig.add_trace(
                go.Scatter(
                    x=wmean_by_w["w"],
                    y=wmean_by_w[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )

        # Maxima on mean
        try:
            idx_mean_max = mean_by_w[m].astype(float).idxmax()
            w_mean_max = int(mean_by_w.loc[idx_mean_max, "w"])
            y_mean_max = float(mean_by_w.loc[idx_mean_max, m])
            fig.add_trace(
                go.Scatter(
                    x=[w_mean_max],
                    y=[y_mean_max],
                    mode="markers",
                    marker=dict(color="green", size=9, symbol="diamond"),
                    name="mean max",
                    showlegend=False,
                )
            )
            try:
                fig.add_vline(x=w_mean_max, line_dash="dash", line_color="green")
            except Exception:
                fig.add_shape(
                    type="line",
                    x0=w_mean_max,
                    x1=w_mean_max,
                    y0=min(mean_by_w[m].astype(float)),
                    y1=max(mean_by_w[m].astype(float)),
                    line=dict(color="green", dash="dash"),
                )
            fig.add_annotation(
                x=w_mean_max,
                y=y_mean_max,
                text=f"mean max: w={w_mean_max}, {m}={y_mean_max:.4f}",
                showarrow=True,
                arrowhead=2,
                ax=40,
                ay=-40,
            )
        except Exception:
            pass

        # Maxima on weighted mean
        try:
            idx_wmean_max = wmean_by_w[m].astype(float).idxmax()
            w_wmean_max = int(wmean_by_w.loc[idx_wmean_max, "w"])
            y_wmean_max = float(wmean_by_w.loc[idx_wmean_max, m])
            fig.add_trace(
                go.Scatter(
                    x=[w_wmean_max],
                    y=[y_wmean_max],
                    mode="markers",
                    marker=dict(color="purple", size=9, symbol="x"),
                    name="weighted mean max",
                    showlegend=False,
                )
            )
            try:
                fig.add_vline(x=w_wmean_max, line_dash="dot", line_color="purple")
            except Exception:
                fig.add_shape(
                    type="line",
                    x0=w_wmean_max,
                    x1=w_wmean_max,
                    y0=min(wmean_by_w[m].astype(float)),
                    y1=max(wmean_by_w[m].astype(float)),
                    line=dict(color="purple", dash="dot"),
                )
            fig.add_annotation(
                x=w_wmean_max,
                y=y_wmean_max,
                text=f"w-mean max: w={w_wmean_max}, {m}={y_wmean_max:.4f}",
                showarrow=True,
                arrowhead=2,
                ax=40,
                ay=-40,
            )
        except Exception:
            pass

        fig.update_layout(
            title=f"{m} (mean/weighted mean across maps) vs sequence length (w)",
            xaxis_title="sequence length (max_window)",
            yaxis_title=m,
        )
        figs[m] = fig
        fig.show()
    return figs


# Results

## Per-map values

In [11]:
# Compute and display aggregated stats, and draw overlays on plots for each map
metrics_cols = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25")
summary_mean_by_w = summary_all.groupby("w")[list(metrics_cols)].mean().reset_index()
summary_weighted_mean_by_w = (
    summary_all
    .groupby("w")
    .apply(lambda g: pd.Series({k: float(np.average(g[k].astype(float), weights=g["num_valid"].astype(float))) for k in metrics_cols}), include_groups=False)
    .reset_index()
)

# Render plots, overlaying mean and weighted mean across all maps
for mname, df_map in summaries.items():
    print(f"\n=== {mname}: mean and weighted-mean overlays ===")
    display(df_map.head(30))
    plot_metrics_vs_window_with_stats(df_map, summary_all)



=== map2: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.875916,0.775680,0.786667,0.822857,0.843810,0.868571,525,605,map2
1,2,0.879031,0.778448,0.790476,0.826667,0.838095,0.866667,525,605,map2



=== map3: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.816202,0.717422,0.934186,0.961609,0.974406,0.985375,547,628,map3
1,2,0.819129,0.719269,0.930530,0.961609,0.979890,0.989031,547,628,map3



=== map4: mean and weighted-mean overlays ===


...

,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.793911,0.684954,0.750779,0.788162,0.820872,0.862928,642,642,map6
1,2,0.800546,0.693241,0.757009,0.791277,0.820872,0.861371,642,642,map6



=== map7: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.785633,0.684605,0.813537,0.892502,0.928334,0.954214,1507,1595,map7
1,2,0.789646,0.687783,0.816191,0.901792,0.932979,0.952887,1507,1595,map7



=== map8: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.778719,0.666404,0.796392,0.858247,0.879725,0.939863,1164,1188,map8
1,2,0.780584,0.668824,0.804124,0.859966,0.880584,0.939863,1164,1188,map8


## Aggregated values

In [12]:
# Display aggregated tables (mean and weighted-mean)
display(summary_mean_by_w.head(30))
try:
    display(summary_weighted_mean_by_w.head(30))
except Exception:
    pass

# Aggregate-only plots (mean and weighted mean across maps) with maxima
plot_aggregate_mean_median(summary_all);


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25
0,1,0.808033,0.69819,0.821244,0.869429,0.895379,0.926725
1,2,0.811610,0.70181,0.824181,0.872254,0.896309,0.926885


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25
0,1,0.799818,0.690543,0.813608,0.868479,0.896336,0.930103
1,2,0.803362,0.694037,0.816985,0.872193,0.897687,0.930103
